# 02b — Seed pipeline with hand-curated Material/

Reads `Train/Material/curated.jsonl` (the hand-curated, `aro check`-
validated examples written by `Train/tools/curate_material.py`) and any
per-prompt training pairs from `Train/Material/<slug>.json` (produced
by `Train/tools/run_prompts.py`), and folds them into the shared
`PAIRS_FILE` via `save_notebook_pair` so EVERY downstream stage picks
them up:

  * NB05 (warm-start fine-tune) — model gets correct ARO from day one
  * NB12 (dataset assembly) → NB12 (SFT)
  * NB12 (preference SFT) — used as `chosen` for matching prompts
  * NB12 (iterative loop) — included in cumulative dataset
  * NB12 (distillation) — teacher trains on the same gold set

NB12 (the Material-focused fine-tune) is the final booster on top.
This notebook ensures the same data is also present *throughout* the
pipeline so every model touch-point benefits.

In [ ]:
import sys
from pathlib import Path
_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import importlib, config
importlib.reload(config)
from config import *

import json
from collections import Counter

MATERIAL_DIR = Path(SCRIPT_DIR).parent / 'Material'
CURATED_FILE = MATERIAL_DIR / 'curated.jsonl'

print(f'Material dir: {MATERIAL_DIR}')
print(f'  curated.jsonl exists: {CURATED_FILE.exists()}')
print(f'  per-prompt JSONs:     {len([p for p in MATERIAL_DIR.glob("*.json") if p.name != "canonical.json"])}')

In [ ]:
# Drop any pairs left over from a previous run so a re-execution doesn't
# duplicate them in PAIRS_FILE.
clean_notebook_pairs('03_material')
clean_notebook_pairs('03_material_runner')

In [ ]:
# 1. Hand-curated examples — already pre-validated by curate.py.
curated_pairs = []
if CURATED_FILE.exists():
    with open(CURATED_FILE) as f:
        for line in f:
            if not line.strip():
                continue
            ex = json.loads(line)
            instr  = ex.get('instruction', '').strip()
            output = ex.get('output', '').strip()
            if not instr or not output:
                continue
            curated_pairs.append({
                'instruction': instr,
                'output':      output,
                'task_type':   ex.get('task_type', 'code_generation'),
                'source':      f'curated/{ex.get("category", "misc")}',
                'score':       1.0,
            })

saved = save_notebook_pairs('03_material', curated_pairs)
print(f'Saved {saved} curated pairs to {PAIRS_FILE}')

In [ ]:
# 2. Per-prompt training pairs from run_prompts.py. Only those with a
#    non-null `expected` field — those are the ones we have ground
#    truth for.
runner_pairs = []
for p in sorted(MATERIAL_DIR.glob('*.json')):
    if p.name == 'canonical.json':
        continue
    try:
        rec = json.loads(p.read_text())
    except json.JSONDecodeError:
        continue
    instr    = (rec.get('prompt') or '').strip()
    expected = (rec.get('expected') or '').strip()
    if not instr or not expected:
        continue
    runner_pairs.append({
        'instruction': instr,
        'output':      expected,
        'task_type':   'code_generation',
        'source':      f'runner/{p.stem}',
        'score':       1.0,
    })

saved = save_notebook_pairs('03_material_runner', runner_pairs)
print(f'Saved {saved} runner-derived pairs to {PAIRS_FILE}')

In [ ]:
totals = Counter()
totals['curated'] = len(curated_pairs)
totals['runner']  = len(runner_pairs)
totals['total']   = sum(totals.values())
for name, n in totals.most_common():
    print(f'  {name:8} {n:>5}')
if totals['total'] == 0:
    print()
    print('No Material pairs found. Run:')
    print('  python3 Train/tools/curate_material.py')
    print('  python3 Train/tools/run_prompts.py')
    print('and re-execute this notebook.')

In [ ]:
# ── Chart: where did the Material pairs come from? ─────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import collections
from pathlib import Path
from datetime import date as _date

plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '03_material_seeding.png'

# Per-category breakdown of the curated set.
_curated_cats = collections.Counter(
    ex.get('category', 'misc') for ex in (
        json.loads(line)
        for line in (MATERIAL_DIR / 'curated.jsonl').read_text().splitlines()
        if line.strip()
    )
)
_categories = [c for c, _ in _curated_cats.most_common()]
_cat_counts = [_curated_cats[c] for c in _categories]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: source pie
_sources = ['curated.jsonl', 'runner JSONs']
_source_counts = [len(curated_pairs), len(runner_pairs)]
_total = sum(_source_counts) or 1
def _pie_autopct(pct):
    n = int(round(pct * _total / 100))
    return f'{n} ({pct:.0f}%)'
ax1.pie(_source_counts, labels=_sources, autopct=_pie_autopct,
        colors=['#3498db', '#2ecc71'], startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax1.set_title(f'Material → pairs.jsonl  ({sum(_source_counts)} pairs)',
              fontsize=12, fontweight='bold')

# Right: curated category breakdown
bars = ax2.barh(_categories, _cat_counts, color='#9b59b6',
                edgecolor='white', height=0.7)
for bar, v in zip(bars, _cat_counts):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
             str(v), va='center', fontsize=9)
ax2.set_xlabel('pairs')
ax2.set_title(f'Curated by category  ({len(curated_pairs)} pairs)',
              fontsize=12, fontweight='bold')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)
ax2.set_axisbelow(True)

fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')
